# **Word2Vec: a Prediction-Based Method**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Idea](#Idea)
- [Objective Function: Negative Log-Likelihood](#Objective-Function:-Negative-Log-Likelihood)
- [How to Train: by Gradient Descent, One Word at a Time](#How-to-Train:-by-Gradient-Descent,-One-Word-at-a-Time)
- [Faster Training: Negative Sampling](#Faster-Training:-Negative-Sampling)
- [Word2Vec Variants: Skip-Gram and CBOW](#Word2Vec-Variants:-Skip-Gram-and-CBOW)
- [Additional Notes](#Additional-Notes)
  - [The Idea is Not New](#The-Idea-is-Not-New)
  - [Why Two Vectors?](#Why-Two-Vectors?)
  - [Better Training](#Better-Training)
  - [The Effect of Window Size](#The-Effect-of-Window-Size)
  - [(Somewhat) Standard Hyperparameters](#(Somewhat)-Standard-Hyperparameters)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Idea

Let us remember our main idea again:

**<u>Main idea</u>: We have to put information about contexts into word vectors**.

While count-based methods took this idea quite literally, Word2Vec uses it in a different manner:

**<u>How</u>: Learn word vectors by teaching them to predict contexts**.

Word2Vec is a model whose parameters are word vectors. These parameters are optimized iteratively for a certain objective. The objective forces word vectors to "know" contexts a word can appear in: the vectors are trained to predict possible contexts of the corresponding words. As you remember from the distributional hypothesis, if vectors "know" about contexts, they "know" word meaning.

**<u>Learned Parameters</u>**: word vectors

**<u>Goal</u>**: make each vector "know" about the contexts of its word

**<u>How</u>**: train vectors to predict possible contexts from words (or, alternatively, words from contexts)

Word2Vec is an iterative method. Its main idea is as follows:
* take a huge text corpus;
* go over the text with a sliding window, moving one word at a time. At each step, there is a central word and context words (other words in this window);
* for the central word, compute probabilities of context words;
* adjust the vectors to increase these probabilities.

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-01.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-02.png" width="49%" />
</p>

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-03.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-04.png" width="49%" />
</p>

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-05.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-06.png" width="49%" />
</p>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Objective Function: Negative Log-Likelihood

For each position $t = 1, 2, ...,T$ in a text corpus, Word2Vec predicts context words within an m-sized window given the central word $\color{#88bd33}{w_t}$:

\begin{equation*}
\color{#88bd33}{\mbox{Likelihood}} \color{black}= L(\theta)=
    \prod\limits_{t=1}^T\prod\limits_{-m\le j \le m, j\neq 0}P(\color{#888}{w_{t+j}}|\color{#88bd33}{w_t}\color{black}, \theta),
\end{equation*}

where $\theta$ are all variables to be optimized. The objective function (aka loss function or cost function) $J(\theta)$ is the average negative log-likelihood:

\begin{equation*}
\color{#88bd33}{\mbox{Loss}} \color{black}= J(\theta) = -\frac{1}{T} \log {L(\theta)} = -\frac {1}{T} \sum_{t=1}^{T} \sum_{\substack{-m \le j \le m \\ j \neq 0}}
 \log P(\color{#888}{w_{t+j}}|\color{#88bd33}{w_t}\color{black}, \theta)
\end{equation*}

Note how well the loss agrees with our main idea above: go over text with a sliding window and compute probabilities. Now let's find out how to compute these probabilities.

**How to calculate $P(\color{#888}{w_{t+j}}\color{black}|\color{#88bd33}{w_t}\color{black}, \theta) ?$**

For each word $w$ we will have two vectors:

* $\color{#88bd33}{v_w}$ when it is a $\color{#88bd33}{\text{central word}}$;
* $\color{#888}{u_w}$ when it is a $\color{#888}{\text{context word}}$.

(Once the vectors are trained, usually we throw away context vectors and use only word vectors.)

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-07.png" width="300">

Then for the central word $\color{#88bd33}{c}$ (c - central) and the context word $\color{#888}{o}$ (o - outside word), the probability of the context word is given below:

\begin{equation*}
P(\color{#888}{o}|\color{#88bd33}{c}) = \frac {exp(\color{#888}{u_o^T} \cdot \color{#88bd33}{v_c})}{\Sigma_{w \in V} exp(\color{#888}{u_w^T} \cdot \color{#88bd33}{v_c})}
\end{equation*}

where:
* $\color{#888}{u_o^T} \cdot \color{#88bd33}{v_c}$ dot product measures the similarity of $\color{#888}{o}$ and $\color{#88bd33}{c}$. Larger dot product => larger probability
* $\Sigma_{w \in V}$ normalizes over entire vocabulary to get probability distribution

**Note**: this is the softmax function!

The function above is an example of the **softmax function**:

\begin{equation*}
\text{softmax}(x_i) = \frac {exp(x_i)}{\sum_{j=i}^n exp(x_j)}
\end{equation*}

**Softmax** maps arbitrary values $x_i$ to a probability distribution $p_i$:

* "**max**" because the largest $x_i$ will have the largest probability $p_i$;
* "**soft**" because all probabilities are non-zero.

You will deal with this function quite a lot over the NLP course (and in Deep Learning in general).

Now, we can go over the illustration. Note that for $\color{#88bd33}{\text{central words}}$ and $\color{#888}{\text{context words}}$, different vectors are used. For example, first the word `a` is central and we use $\color{#88bd33}{v_a}$, but when it becomes context, we use $\color{#888}{u_a}$ instead.

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-08.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-09.png" width="49%" />
</p>

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-10.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-11.png" width="49%" />
</p>

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-12.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-13.png" width="49%" />
</p>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# How to Train: by Gradient Descent, One Word at a Time

Let us recall that our parameters $\theta$ are vectors $\color{#88bd33}{v_w}$ and $\color{#888}{u_w}$ for all words in the vocabulary. These vectors are learned by optimizing the training objective via gradient descent (with some learning rate $\alpha$):

\begin{equation*}
\theta^{new} = \theta^{old} - \alpha \nabla_{\theta} J(\theta)
\end{equation*}

**<u>One word at a time</u>**

We make these updates one at a time: each update is for a single pair of a center word and one of its context words. Look again at the loss function:

\begin{equation*}
\color{#88bd33}{\mbox{Loss}} \color{black}= J(\theta) = -\frac{1}{T} \log {L(\theta)} = -\frac {1}{T} \sum_{t=1}^{T} \sum_{\substack{-m \le j \le m \\ j \neq 0}}
 \log P(\color{#888}{w_{t+j}}|\color{#88bd33}{w_t}\color{black}, \theta) = \frac {1}{T} \sum_{t=1}^{T} \sum_{\substack{-m \le j \le m \\ j \neq 0}} J_{t,j}(\theta).
\end{equation*}

For the center word $\color{#88bd33}{w_t}$, the loss contains a distinct term $J_{t,j}(\theta) = -\log P(\color{#888}{w_{t+j}}|\color{#88bd33}{w_t}\color{black}, \theta)$ for each of its context words $\color{#888}{w_{t+j}}$. Let us look in more detail at just this one term and try to understand how to make an update for this step. For example, let's imagine we have a sentence:

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-14.png" width="400" />

with the central word $\color{#88bd33}{\text{cat}}$, and four context words. Since we are going to look at just one step, we will pick only one of the context words; for example, let's take $\color{#888}{\text{cute}}$. Then the loss term for the central word $\color{#88bd33}{\text{cat}}$ and the context word $\color{#888}{\text{cute}}$ is:

\begin{equation*}
J_{t,j}(\theta)= -\log P(\color{#888}{cute}\color{black}|\color{#88bd33}{cat}\color{black}) =
        -\log \frac{\exp\color{#888}{u_{cute}^T}\color{#88bd33}{v_{cat}}}{
       \sum\limits_{w\in Voc}\exp{\color{#888}{u_w^T}\color{#88bd33}{v_{cat}} }} =
    -\color{#888}{u_{cute}^T}\color{#88bd33}{v_{cat}}\color{black}
        + \log \sum\limits_{w \in Voc}\exp{\color{#888}{u_w^T}\color{#88bd33}{v_{cat}}}\color{black}{.}
\end{equation*}

Note which parameters are present at this step:

* from vectors for $\color{#88bd33}{\text{central words}}$, only $\color{#88bd33}{v_{cat}}$;
* from vectors for $\color{#888}{\text{context words}}$, all $\color{#888}{v_w}$ (for all words in the vocabulary).

Only these parameters will be updated at the current step.

Below is the schematic illustration of the derivations for this step.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-15.png" width="600" />

By making an update to minimize $J_{t,j}(\theta)$, we force the parameters to **<u>increase</u>** similarity (dot product) of $\color{#88bd33}{v_{cat}}$ and $\color{#888}{u_{cute}}$ and, at the same time, to **<u>decrease</u>** similarity between $\color{#88bd33}{v_{cat}}$ and $\color{#888}{u_{w}}$ for all other words $w$ in the vocabulary.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-16.png" width="400" />

This may sound a bit strange: why do we want to decrease similarity between $\color{#88bd33}{v_{cat}}$ and all other words, if some of them are also valid context words (e.g., $\color{#888}{\text{grey}}$, $\color{#888}{\text{playing}}$, $\color{#888}{\text{in}}$, on our example sentence)?

But do not worry: since we make updates for each context word (and for all central words in your text), <u>on average over all updates</u> our vectors will learn the distribution of the possible contexts.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Faster Training: Negative Sampling

In the example above, for each pair of a central word and its context word, we had to update all vectors for context words. This is highly inefficient: for each step, the time needed to make an update is proportional to the vocabulary size.

But why do we have to consider **all** context vectors in the vocabulary at each step? For example, imagine that at the current step we consider context vectors not for all words, but only for the current target ($\color{#888}{\text{cute}}$) and several randomly chosen words. The figure shows the intuition.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-17.png" width="600"/>

As before, we are increasing similarity between $\color{#88bd33}{v_{cat}}$ and $\color{#888}{u_{cute}}$. What is different, is that now we decrease similarity between $\color{#88bd33}{v_{cat}}$ and context vectors <u>not for all words</u>, but only with a <u>subset of K "negative" examples</u>.

Since we have a large corpus, on average over all updates we will update each vector sufficient number of times, and the vectors will still be able to learn the relationships between words quite well.

Formally, the new loss function for this step is:

\begin{equation*}
J_{t,j}(\theta)=
    -\log\sigma(\color{#888}{u_{cute}^T}\color{#88bd33}{v_{cat}}\color{black}) -
    \sum\limits_{w\in \{w_{i_1},\dots, w_{i_K}\}}\log\sigma({-\color{#888}{u_w^T}\color{#88bd33}{v_{cat}}}\color{black}),
\end{equation*}

where $w_{i_1},\dots, w_{i_K}$ are the K negative examples chosen at this step and $\sigma(x)=\frac{1}{1+e^{-x}}$ is the sigmoid function.

Note that $\sigma(-x)=\frac{1}{1+e^{x}}=\frac{1\cdot e^{-x}}{(1+e^{x})\cdot e^{-x}} = \frac{e^{-x}}{1+e^{-x}}= 1- \frac{1}{1+e^{-x}}=1-\sigma(x)$. Then the loss can also be written as:

\begin{equation*}
J_{t,j}(\theta)=
    -\log\sigma(\color{#888}{u_{cute}^T}\color{#88bd33}{v_{cat}}\color{black}) -
    \sum\limits_{w\in \{w_{i_1},\dots, w_{i_K}\}}\log(1-\sigma({\color{#888}{u_w^T}\color{#88bd33}{v_{cat}}}\color{black})).
\end{equation*}

<u>The Choice of Negative Examples</u>

Each word has only a few "true" contexts. Therefore, randomly chosen words are very likely to be "negative", i.e. not true contexts. This simple idea is used not only to train Word2Vec efficiently but also in many other applications, some of which we will see later in the course.

Word2Vec randomly samples negative examples based on the empirical distribution of words. Let $U(w)$ be a unigram distribution of words, i.e. $U(w)$ is the frequency of the word $w$ in the text corpus. Word2Vec modifies this distribution to sample less frequent words more often: it samples proportionally to $U^{3/4}(w)$.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Word2Vec Variants: Skip-Gram and CBOW

There are two Word2Vec variants: Skip-Gram and CBOW.

* Skip-Gram is the model we considered so far: it predicts context words given the central word. Skip-Gram with negative sampling is the most popular approach.
* CBOW (Continuous Bag-of-Words) predicts the central word from the sum of context vectors. This simple sum of word vectors is called "bag of words", which gives the name for the model.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-18.png" width="600"/>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Additional Notes

The original Word2Vec papers are:

* [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/pdf/1301.3781.pdf)
* [Distributed Representations of Words and Phrases and their Compositionality](https://papers.nips.cc/paper/5021-distributed-representations-of-words-and-phrases-and-their-compositionality.pdf)

You can look into them for the details on the experiments, implementation and hyperparameters. Here we will provide some of the most important things you need to know.

## The Idea is Not New

The idea to learn word vectors (distributed representations) is not new. For example, there were attempts to learn word vectors as part of a larger network and then extract the embedding layer. (For the details on the previous methods, you can look, for example, at the summary in the original Word2Vec papers).

What was very unexpected in Word2Vec, is its ability to learn <u>high-quality</u> word vectors <u>very fast</u> on huge datasets and for large vocabularies.

## Why Two Vectors?

As you remember, in Word2Vec we train two vectors for each word: one when it is a central word and another when it is a context word. After training, context vectors are thrown away.

This is one of the tricks that made Word2Vec so simple. Look again at the loss function (for one step):

\begin{equation*}
J_{t,j}(\theta)=
    -\color{#888}{u_{cute}^T}\color{#88bd33}{v_{cat}}\color{black} +
    \log \sum\limits_{w\in V}\exp{\color{#888}{u_w^T}\color{#88bd33}{v_{cat}}}\color{black}{.}
\end{equation*}

When central and context words have different vectors, both the first term and dot products inside the exponents are linear with respect to the parameters (the same for the negative training objective). Therefore, the gradients are easy to compute.

After training GloVe averages word and context vectors, while Word2Vec throws context vectors away. While this is the standard practice, it was shown that sometimes Word2Vec can also benefit from averaging word and context vectors. For more info, refer paper [Improving Distributional Similarity with Lessons Learned from Word Embeddings](https://aclanthology.org/Q15-1016.pdf).

## Better Training

**<u>Are all context words equally important for training?</u>**

During Word2Vec training, we make an update for each of the context words. For example, for the central word $\color{#88bd33}{\text{cat}}$ we make an update for each of the words $\color{#888}{\text{cute}}$, $\color{#888}{\text{grey}}$, $\color{#888}{\text{playing}}$, $\color{#888}{\text{in}}$.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-19.png" width="400"/>

Which word types give more/less information than others? Think about some characteristics of words that can influence their importance.

* **Word frequency**: We can expect that frequent words usually give less information than rare ones. For example, the fact that $\color{#88bd33}{\text{cat}}$ appears in context of $\color{#888}{\text{in}}$ does not tell us much about the meaning of $\color{#88bd33}{\text{cat}}$: the word $\color{#888}{\text{in}}$ serves as a context for many other words. In contrast, $\color{#888}{\text{cute}}$, $\color{#888}{\text{grey}}$ and $\color{#888}{\text{playing}}$ already give us some idea about $\color{#88bd33}{\text{cat}}$.

* **Distance from the central word:** Words that are closer to the central word may be more important.

**<u>How can we use this to modify training?</u>**

Tricks from the original Word2Vec paper are given below:

1. **Word Frequency**:

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-20.png" width="500"/>

    To account for different informativeness of rare and frequent words, Word2Vec uses a simple subsampling approach: each word $w_i$ in the training set is ignored with probability computed by the formula

    \begin{equation*}
    P(w_i)=1 - \sqrt{\frac{\text{thr}}{f(w_i)}}
    \end{equation*}

    where $f(w_i)$ is the word frequency and $\text{thr}$ is the chosen threshold (in the original paper, $\text{thr} = 10^{-5}$). This formula preserves the ranking of the frequencies, but aggressively subsamples words whose frequency is greater than $\text{thr}$.

    Interestingly, this heuristic works well in practice: it accelerates learning and even significantly improves the accuracy of the learned vectors of the rare words.

2. **Distance from the central word**:

    We can assign higher weights to the words which are closer to the central word.

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-21.png" width="400"/>

    At the first glance, you won't see any weights in the original Word2Vec implementation. However, at each step it samples the size of the context window from 1 to L. Therefore, words which are closer to central are used more frequently than the distant ones. In the original work this was (probably) done for efficiency (fewer updates for each step), but this also has the effect similar to assigning weights.

## The Effect of Window Size

The size of the sliding window has a strong effect on the resulting vector similarities. For example, [this paper](https://arxiv.org/pdf/1510.00726) notes that larger windows tend to produce more topical similarities (i.e. dog, bark and leash will be grouped together, as well as walked, run and walking), while smaller windows tend to produce more functional and syntactic similarities (i.e. Poodle, Pitbull, Rottweiler, or walking, running, approaching).

## (Somewhat) Standard Hyperparameters

As always, the choice of hyperparameters usually depends on the task at hand; you can look at the original papers for more details.

Somewhat standard setting is:
* **Model**: Skip-Gram with negative sampling;
* **Number of negative examples**: for smaller datasets, 15-20; for huge datasets (which are usually used) it can be 2-5.
* **Embedding dimensionality**: frequently used value is 300, but other variants (e.g., 100 or 50) are also possible.
* **Sliding window (context) size**: 5-10.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[NLP Course by Lena Voita](https://lena-voita.github.io/nlp_course.html)**
- **Key Papers:**
    - [Word2Vec Paper, 2013](https://arxiv.org/pdf/1301.3781)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)